# 🪨 StoryStone — The Curious Heritage Detective

> **Every sculpture has a story. StoryStone investigates the clues and discovers it.**

This notebook documents the full StoryStone project built for the
[Agent Experience Hackathon: Strands Decider](https://builder.aws.com/build/hackathons)
(AWS User Group Madurai, October 2026).

**Tags:** `#strands-decider-2b` `#policy`

---

## Table of Contents
1. [Project Overview](#1-project-overview)
2. [Architecture](#2-architecture)
3. [The Strands Decider 2B Decision](#3-the-strands-decider-2b-decision)
4. [Tools & Data](#4-tools--data)
5. [Live Demo — Temple Introduction](#5-live-demo--temple-introduction)
6. [Live Demo — Sculpture Investigation](#6-live-demo--sculpture-investigation)
7. [Decision Deep Dive](#7-decision-deep-dive)
8. [Benchmark Results](#8-benchmark-results)
9. [Reusable Decision Pattern](#9-reusable-decision-pattern)
10. [Deployment](#10-deployment)
11. [What's Next](#11-whats-next)

## 1. Project Overview

### The Problem
Millions of tourists walk past extraordinary temple sculptures every day without understanding
what they're looking at. A four-armed dancing figure with a ring of fire — Nataraja? An
elephant-headed deity — Ganesha? A blurry figure whose upper arms are worn smooth — Vishnu or Murugan?

Existing AI tools **describe** images. StoryStone **investigates** them:
- It researches the temple context
- It analyses iconographic details against a heritage knowledge base
- It searches for matching mythology and references
- It asks Strands Decider 2B whether the evidence is *sufficient* before presenting any story
- It honestly admits uncertainty rather than fabricating an answer

### The User Journey
```
Tourist enters: "Meenakshi Amman Temple, Madurai"
    ↓
StoryStone presents: temple history, legends, highlights, sources
    ↓
Tourist types: "Dancing figure, four arms, ring of fire, small dwarf below"
    ↓
StoryStone investigates: analyses, searches, assembles evidence
    ↓
Strands Decider 2B: verdict = "answer" (p=0.91)
    ↓
StoryStone presents: Nataraja — the cosmic dance of Shiva, with sources
```

## 2. Architecture

```
Demo UI (FastAPI + HTML)
    │
    └── HackathonAgent.chat(message, images)
            │
            ├── [Temple branch] ─────────────────────────────────────
            │   research_temple(name, city)   ← DuckDuckGo scrape
            │   LLM call #1: temple intro narration (ugmdu / Bedrock)
            │
            └── [Sculpture branch] ──────────────────────────────────
                ├── [Optional] Ollama qwen3.5:4b: image → text description
                ├── analyze_sculpture(description, temple_ctx)  ← mock DB
                ├── search_heritage_references(candidate, ...)  ← DuckDuckGo
                ├── evaluate_evidence(...)                      ← text assembly
                │
                ├── Strands Decider 2B ─── verdict + evidence_quality + needs_visual
                │       (hosted API, ~1–2 s, 3 questions, 1 forward pass)
                │
                │   [if investigate_further: search again, re-evaluate, re-decide]
                │
                └── LLM call #2: story narration (ugmdu / Bedrock Nova Pro)
```

**Key numbers:**
- LLM calls per investigation: **2**
- Strands Decider 2B calls: **1** (3 questions, shared encoding)
- DuckDuckGo searches: **1–2** (no API key needed)
- Rate limit exposure: **2 LLM calls** (vs 5–8 in a naive agentic loop)

## 3. The Strands Decider 2B Decision

**Pattern:** `#policy` — Evidence sufficiency classification

The decider reads the assembled evidence text (plain English prose) and classifies
the investigation verdict. This is a *language judgement* — whether references match
observations, whether ambiguities are critical, whether sources contradict.

### Why not an LLM judge?
| | Strands Decider 2B | LLM judge |
|---|---|---|
| Latency | ~1–2 s | ~3–5 s |
| Cost | ~$0.000 (free tier) | ~$0.002–0.005 |
| Output | Calibrated probabilities | Text that says 'fairly confident' |
| Tuneable | Yes — retrain questions | No — reprompt |
| Testable | Yes — labelled eval cases | Harder to measure |

### State text layout
```
=== TEMPLE CONTEXT ===
Temple: Meenakshi Amman Temple, Madurai
Background: [temple summary]

=== SCULPTURE OBSERVATIONS ===
[posture, mudras, attributes, vahana, ornaments]

=== CANDIDATE IDENTIFICATION ===
[e.g. Nataraja — Shiva as Lord of Cosmic Dance]

=== RETRIEVED HERITAGE REFERENCES ===
[web search results with URLs]

=== INVESTIGATION QUESTION ===
[tourist's question]
```

### Verdict actions
```python
if verdict.action == 'answer':               present sourced story
if verdict.action == 'investigate_further':  search again, re-evaluate
if verdict.action == 'request_closeup':      ask for a focused photo
if verdict.action == 'acknowledge_uncertainty': present observations only

if verdict.low_confidence:  # confidence < 0.55
    reply += '⚠️ tentative interpretation'
```

In [ ]:
# Setup: add project root to path
import sys, os
from pathlib import Path

# Adjust this path to your kit location
KIT_ROOT = Path(".").resolve()
sys.path.insert(0, str(KIT_ROOT))
os.chdir(KIT_ROOT)

print(f"Kit root: {KIT_ROOT}")
print(f"Python: {sys.version.split()[0]}")

In [ ]:
# Inspect the decision pattern
from agent.decision import PATTERN_TAG, QUESTIONS, InvestigationPolicy

print(f"Pattern tag: {PATTERN_TAG}")
print(f"\nQuestions ({len(QUESTIONS)}):")
for name, q in QUESTIONS.items():
    print(f"  {name} ({q['type']}): {q['instructions'][:80]}...")
    if q['type'] == 'choice':
        for option, desc in q['criteria'].items():
            print(f"    → {option}: {desc[:60]}...")

## 4. Tools & Data

| Tool | Purpose | Data source |
|---|---|---|
| `research_temple(name, city)` | Temple history, legends, highlights, source URLs | DuckDuckGo HTML scrape (no API key) |
| `analyze_sculpture(description, ctx)` | Iconographic observations, candidate ID | Mock heritage DB (`agent/data/sculptures.json`) |
| `search_heritage_references(candidate, ...)` | Mythology, iconography, academic sources | DuckDuckGo HTML scrape |
| `evaluate_evidence(...)` | Assembles structured state text for decider | Local text assembly |

### Mock data
`agent/data/sculptures.json` contains rich iconographic entries for:
- **Nataraja** — Shiva as Lord of Cosmic Dance
- **Ganesha** — elephant-headed son of Shiva and Parvati
- **Murugan** — Tamil god of war and divine knowledge
- **Dakshinamurthy** — Shiva as the silent teacher
- **Ambiguous figure** — deliberately unclear (triggers `request_closeup`)

Each entry includes: observations, iconographic notes, and 3 reference URLs.

In [ ]:
# Inspect the tools
from agent.tools import research_temple, analyze_sculpture, search_heritage_references, evaluate_evidence
import json

# Test analyze_sculpture with a Nataraja description
result = analyze_sculpture(
    image_description="dancing figure four arms ring of fire",
    temple_context="Meenakshi Amman Temple, Madurai"
)
print("analyze_sculpture result:")
print(f"  candidate: {result['candidate']}")
print(f"  needs_closeup: {result['needs_closeup']}")
print(f"  observations (first 200 chars): {result['observations'][:200]}")

In [ ]:
# Test evidence assembly
ev = evaluate_evidence(
    temple_name="Meenakshi Amman Temple",
    temple_city="Madurai",
    temple_summary="Major Shaivite temple complex in Madurai, Tamil Nadu.",
    observations=result['observations'],
    candidate=result['candidate'],
    references="https://en.wikipedia.org/wiki/Nataraja: Nataraja is Shiva as the cosmic dancer.",
    user_question="Who is this dancing figure?"
)
print("State text sent to Strands Decider 2B:")
print("=" * 60)
print(ev['state_text'])

## 5. Live Demo — Temple Introduction

This cell makes a **real call** to the AWS User Group Madurai cloud LLM.
Requires a valid `DECIDER_API_KEY` in your `.env`.
Latency: ~15–30 s on the user group cloud.

In [ ]:
from decider_kit.config import load_dotenv
load_dotenv()  # load .env

from agent.agent import HackathonAgent

agent = HackathonAgent()
print(f"LLM: {agent.model_id}")
print()

r = agent.chat(
    "I'm visiting Meenakshi Amman Temple in Madurai. Tell me about this temple.",
    session_id="notebook-demo"
)
print(r['reply'])
print(f"\n[{r['latency_ms']} ms | {len(r['decisions'])} decisions]")

## 6. Live Demo — Sculpture Investigation

This cell makes a real call through the full pipeline:
- `analyze_sculpture` (mock data lookup)
- `search_heritage_references` (DuckDuckGo)
- `evaluate_evidence` (text assembly)
- Strands Decider 2B (hosted API)
- LLM story narration (user group cloud)

In [ ]:
import json

r2 = agent.chat(
    "I found a figure with four arms performing a dance, surrounded by a ring of fire, "
    "standing on a small dwarf. There's a drum in one upper hand and a flame in the other.",
    session_id="notebook-demo"
)

print("=" * 60)
print("STORY:")
print("=" * 60)
print(r2['reply'])
print()
print("=" * 60)
print("STRANDS DECIDER 2B DECISIONS:")
print("=" * 60)
for d in r2['decisions']:
    print(json.dumps(d, indent=2))
print(f"\n[{r2['latency_ms']} ms total]")

## 7. Decision Deep Dive

Let's run the decision directly with a `FakeDecider` to understand all four verdict paths.

In [ ]:
import sys
sys.path.insert(0, str(KIT_ROOT / 'tests'))
from fakes import FakeDecider
from agent.decision import InvestigationPolicy

SAMPLE_STATE = """=== TEMPLE CONTEXT ===
Temple: Brihadeeswarar Temple, Thanjavur
Background: UNESCO World Heritage Site, dedicated to Shiva, built by Raja Raja Chola I.

=== SCULPTURE OBSERVATIONS ===
Four-armed figure in Ananda Tandava. Damaru upper right. Flame upper left.
Abhaya mudra lower right. Apasmara dwarf at base. Ring of flames.

=== CANDIDATE IDENTIFICATION ===
Nataraja — Shiva as the Lord of Cosmic Dance

=== RETRIEVED HERITAGE REFERENCES ===
https://en.wikipedia.org/wiki/Nataraja: All five canonical attributes present.
https://www.metmuseum.org/art/...: Chola bronze with drum, fire, abhaya mudra.

=== INVESTIGATION QUESTION ===
Who is this dancing figure?"""

verdicts = ['answer', 'investigate_further', 'request_closeup', 'acknowledge_uncertainty']

for expected_verdict in verdicts:
    def rule(state, name, q, v=expected_verdict):
        if name == 'verdict': return v
        if name == 'evidence_quality': return 0.85 if v == 'answer' else 0.2
        return 0.8 if v == 'request_closeup' else 0.1  # needs_visual_detail
    
    policy = InvestigationPolicy(FakeDecider(rule))
    verdict = policy.check(SAMPLE_STATE)
    print(f"Decider says '{expected_verdict}' → action: '{verdict.action}' | "
          f"confidence: {verdict.verdict_confidence:.2f} | "
          f"low_confidence: {verdict.low_confidence}")

In [ ]:
# Show the evidence_quality override:
# If decider says 'answer' but evidence_quality p_yes < 0.40 → override to 'acknowledge_uncertainty'

def low_evidence_rule(state, name, q):
    if name == 'verdict': return 'answer'       # decider wants to answer...
    if name == 'evidence_quality': return 0.15  # ...but evidence is too thin!
    return 0.1

policy = InvestigationPolicy(FakeDecider(low_evidence_rule))
v = policy.check(SAMPLE_STATE)
print(f"Decider said 'answer' but evidence_quality=0.15 → action overridden to: '{v.action}'")
print("This is the safety net: we never present a story when evidence is too thin.")

## 8. Benchmark Results

The kit's `./kit bench` command runs Strands Decider 2B against an LLM baseline on the same
24 labelled eval cases. Results are saved to `submission/bench/`.

**Eval case distribution:**
- `answer`: 8 cases (strong evidence, clear match)
- `investigate_further`: 4 cases (thin evidence, needs more search)
- `request_closeup`: 4 cases (blurry/damaged key attribute)
- `acknowledge_uncertainty`: 4 cases (absent/contradictory references)
- Near-miss cases: 4 (e.g. answer with evidence_quality < 0.4 → override)

In [ ]:
# Load benchmark results if they exist
import json
from pathlib import Path

results_path = KIT_ROOT / 'submission' / 'bench' / 'results.json'
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print("Benchmark results:")
    print(json.dumps(results, indent=2))
else:
    print("Benchmark not run yet. Run: .\\kit.cmd bench --split tune")
    print("Then: .\\kit.cmd bench --split test")
    print("Results will appear in submission/bench/results.json")

In [ ]:
# Preview eval cases
import json
from pathlib import Path
from collections import Counter

cases = [json.loads(l) for l in (KIT_ROOT / 'agent' / 'eval_cases.jsonl').read_text().splitlines() if l.strip()]
print(f"Total eval cases: {len(cases)}")

splits = Counter(c.get('split', 'unknown') for c in cases)
verdicts = Counter(c['expected'].get('verdict', '?') for c in cases)
print(f"Splits: {dict(splits)}")
print(f"Verdict distribution: {dict(verdicts)}")

print("\nFirst 3 cases (state preview + expected):")
for c in cases[:3]:
    state_preview = c['state'][:120].replace('\n', ' ')
    print(f"  [{c.get('split')}] expected={c['expected']} | state: {state_preview}...")

## 9. Reusable Decision Pattern

The `#policy` pattern in `agent/decision.py` is designed to be copied into any agent
that needs to classify whether assembled evidence or content is sufficient to act on.

### How to reuse it

1. Copy `agent/decision.py`
2. Rewrite the four `verdict` option descriptions for your domain
3. Adjust thresholds in `InvestigationPolicy.decide()`
4. Call `policy.check(your_state_text)` wherever you need the verdict

### Example reuse cases

| Domain | Verdicts |
|---|---|
| Claims verification | `verified` / `needs_more_evidence` / `contradicted` / `uncertain` |
| Content moderation | `publish` / `review` / `reject` / `needs_context` |
| Research assistant | `report_finding` / `search_more` / `flag_contradiction` / `inconclusive` |
| Medical triage | `refer_to_specialist` / `routine_followup` / `urgent` / `insufficient_info` |

In [ ]:
# Show the reusable pattern docstring
import ast
from pathlib import Path

src = (KIT_ROOT / 'agent' / 'decision.py').read_text(encoding='utf-8')
tree = ast.parse(src)
docstring = ast.get_docstring(tree)
print(docstring)

## 10. Deployment

The submitted agent runs on **Amazon Bedrock AgentCore Runtime** with **Amazon Nova Pro**
as the generative LLM. The Strands Decider 2B hosted API (AWS User Group Madurai) is used
for the policy decision.

### Deployment steps

```powershell
# 1. Install tools
.\kit.cmd tools install deploy --yes

# 2. Add AWS credentials
.\kit.cmd aws-login

# 3. Enable Amazon Nova Pro in Bedrock Model Access (AWS Console)
#    Bedrock → Model access → Amazon Nova Pro → Request access

# 4. Deploy (~3 min)
.\kit.cmd deploy

# 5. Verify
.\kit.cmd invoke "I'm at Meenakshi Amman Temple in Madurai. Tell me about this temple."

# 6. Record demo video
.\kit.cmd video --target agentcore

# 7. Clean up (after judging)
.\kit.cmd destroy
```

### Cost estimate
- AgentCore Runtime: pay per request-second (cents for demo traffic)
- Nova Pro: ~$0.002–0.005 per investigation turn
- Total for hackathon demo: **< $1**

## 11. What's Next

### Immediate (pre-submission)
- [ ] Get AWS credentials and run `./kit deploy`
- [ ] Install `gh` CLI and run `./kit publish <repo-name>`
- [ ] Record demo video: `./kit video --target agentcore`
- [ ] Publish article on AWS Builder Center
- [ ] Submit before 23 October 2026, 11:59 PM PT

### Future improvements
- **Real vision analysis** — switch from mock iconographic DB to a multimodal Bedrock model
  (Nova Lite is vision-capable and costs ~$0.001 per image)
- **More sculptures** — expand `agent/data/sculptures.json` to cover more deities and traditions
- **AgentCore Memory** — remember which sculptures a tourist has investigated across sessions
- **Multilingual** — present stories in Tamil, Hindi, or the tourist's language
- **Offline mode** — pre-cache heritage references for popular temples

---

Built with the **AWS User Group Madurai** Strands Decider hackathon kit.

Visit [awsugmdu.in](https://www.awsugmdu.in) for the community and events.